### Fix zotero database errors and annoyances

Looks for entries with dead

- *warning*: this won't handle items with more than one attachment of the same filetype, so I'm skipping them
- *warning*: this currently deletes files with 'htm' extensions instead of making them `html'
   - **Need to fix** b/c SingleFile saves are always .htm

In [1]:
from icecream import ic
import pathlib as pl
from pyzotero import zotero
import pandas as pd
from IPython.display import display, HTML

import pathlib as pl
from icecream import ic
import sys

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [ ]:
zot = zotero.Zotero(rfw.zotero_library_id, rfw.zotero_library_type, rfw.zotero_api_key)
parents = zot.everything(zot.top())

In [3]:
#doFixes = False # if True actually rename files and update the database when there are errors
doFixes = True # if True actually rename files and update the database when there are errors
oddities = []
for parent in parents: # shortParents:
    pkey = parent['key']
    parentCitekey=rfw.get_citation_key(parent['data'])
    id_dict = dict(parentCitekey=parentCitekey, parentKey=pkey)

    # Avoid complexity of more than one child of same file extension
    attachments_data = pd.DataFrame([child['data'] for child in zot.children(pkey)])
    if 'contentType' in attachments_data.columns and 'path' in attachments_data.columns:
        typeCounts = attachments_data.query('path.isna()').groupby('contentType').count()['key']
        if any(typeCounts > 1):
            countString = typeCounts.to_string(header=False).replace('\n', ' ')
            oddities.append(id_dict | {'message':f'Skipping parent w/ >1 attachments of same type: Counts: {countString}'})

    try:
        children = rfw.get_children(pkey,zot)
    except:
        oddities.append(id_dict | {'message':f'Skipping since failed to get children of this parent'})
        continue

    for child in children:
        if rfw.is_ignorable_child(child):
           continue

        cdat = child['data']
        childContentType = cdat['contentType']
        id_dict = id_dict | dict(childKey=cdat['key'], childContentType=childContentType)

        try:
            desiredExt = rfw.desiredFileExtention[childContentType]
        except:
            oddities.append(id_dict | dict(message=f"Unhandled attachment {childContentType=}"))
            continue # skip: many are dead .md files from my BetterNotes experiments

        foundAttachPath = cdat['path'].removeprefix("attachments:")
        desiredBaseFNm = f'{parentCitekey}.{desiredExt}'
        desired_file_full_path = rfw.lit_attachment_dir_shared / desiredBaseFNm

        titleBadFixable = False # False: title is already OK or not fixed b/c a bad filename couldn't be fixed
        if (titleBad := cdat['title'] != desiredBaseFNm):
            child['data']['title'] = desiredBaseFNm # potential title fix
            titleBadFixable = True  # maybe, but don't fix if a bad file path couldn't have been fixed

        
        filePathBadFixable, filePathBadFixed = False, False
        if (filePathBad := not desired_file_full_path.exists()):
            if (guess_file_full_path := rfw.lit_attachment_dir_shared / foundAttachPath).exists():                
                # file is there but is misnamed: fix it
               filePathBadFixable = True # probably fixable
               child['data']['path'] = f'attachments:{desiredBaseFNm}' # potential file path fix
               if doFixes:
                    try: 
                        guess_file_full_path.rename(desired_file_full_path)
                        filePathBadFixed = True
                    except:
                        print(f'{pkey}: failed to rename {str(guess_file_full_path)}')
                        filePathBadFixable = False # since rename failed, don't update path in DB
                        titleBadFixable = False # don't change title either

        dbNeedsUpdate = titleBadFixable or filePathBadFixable
        dbUpdated = False
        if (dbUpdateAttempted := (doFixes and dbNeedsUpdate)):
            try: 
                #child_before_update = child.copy()
                zot.update_item(child)
                dbUpdated = True
                # updated_children = list(zot.children(pkey))
                # raise Exception(f'{pkey=}, {child['key']=}: An update was attempted.  See if it worked!')
            except:
                raise Exception(f'{pkey}: failed to update database')

        if titleBad or filePathBad:
            oddities.append(id_dict | dict(titleBad=titleBad, 
                                           filePathBad=filePathBad,
                                           filePathBadFixable=filePathBadFixable,
                                           filePathBadFixed=filePathBadFixed,
                                           titleBadFixable=titleBadFixable,
                                           message='Bad Attachment',
                                           foundAttachPath=foundAttachPath,
                                           dbNeedsUpdate=dbNeedsUpdate,
                                           dbUpdateAttempted=dbUpdateAttempted,
                                           dbUpdated=dbUpdated))

In [4]:
oddities = pd.DataFrame(oddities)
print(f'{len(oddities)} attachments had problems')
pd.DataFrame(oddities[['titleBad', 'filePathBad']].sum())

badPathRows = oddities[oddities.filePathBad]
fixableBadPathRows = badPathRows[badPathRows.filePathBadFixable]
fixedBadPathRows = badPathRows[badPathRows.filePathBadFixed]
print(f'{len(fixableBadPathRows)} of {len(badPathRows)} bad paths are fixable: {len(fixedBadPathRows)} were fixed.')

nonFixableBadPathRows = badPathRows[~badPathRows.filePathBadFixable]
print(f'{len(nonFixableBadPathRows)} non fixable bad filepaths found')
display(HTML('<b>Non-fixable</b>'))
nonFixableBadPathRows

1672 attachments had problems


KeyError: "None of [Index(['titleBad', 'filePathBad'], dtype='object')] are in the [columns]"

In [ ]:
oddities


In [ ]:
unfixableAttachments = []
for ix, row in nonFixableBadPathRows.iterrows():
    row.childKey, row.parentCitekey, row.parentKey
    child = zot.item(row.childKey)
    parent = zot.item(row.parentKey)

    row['parentURL'] = parent['data']['url']
    row['childContentType'] = child['data']['contentType']

    if (contentGettable := row['childContentType'] == 'text/html'):
    
    

    #for child in zot.children(pkey):
    #ic(row['childKey'])
    #ic(type(row))

In [ ]:
if 

